

# NLP Pandas Hugging Face (2)
### OPIM 5512 — Applied Data Science · Module5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module5/NLP_Pandas_HuggingFace (2).ipynb)

*Run me top to bottom — **Runtime → Run all**. Data loads from a stable link, so there's nothing to upload.*

# Downloading pre-trained Transformers and running them locally for text analytics
------------------------------
**Dr. Dave Wanik - Dept. of Operations and Information  - University of Connecticut**

Go grab some incredible pre-trained models off the shelf that have been curated for specific tasks (like summarization.) These models do not generate text from scratch - they generate text conditioned on an input sequence.

In [ ]:
!pip install ipython-autotime
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 8.95 s (started: 2025-12-12 11:34:47 +00:00)


In [ ]:
#!pip install pyspark
!pip install transformers datasets evaluate rouge_score

time: 39.2 s (started: 2025-12-12 11:34:56 +00:00)


# 🤗 Hugging Face is amazing - let's get the toy Billsum data

# Read data

In [ ]:
from datasets import load_dataset

billsum = load_dataset("billsum", split="ca_test")

time: 1.69 s (started: 2025-12-12 11:35:36 +00:00)


In [ ]:
# attributes of the dataset
billsum

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 1237
})

time: 14.1 ms (started: 2025-12-12 11:35:37 +00:00)


In [ ]:
# convert to a pandas dataframe for something familiar
# you could always save as a .csv and load dataframe
import pandas as pd
df = pd.DataFrame(billsum)
df.head()

,text,summary,title
0,The people of the State of California do enact...,Existing property tax law establishes a vetera...,An act to amend Section 215.1 of the Revenue a...
1,The people of the State of California do enact...,Existing law provides that the Board of Parole...,"An act to amend Section 3550 of, and to add Se..."
2,The people of the State of California do enact...,The Sales and Use Tax Law imposes a tax on ret...,An act\nto add Chapter 3.8 (commencing with Se...
3,The people of the State of California do enact...,"Existing law requires all moneys, except for f...","An act to amend Sections 75220, 75221, and 752..."
4,The people of the State of California do enact...,Existing law defines a request regarding resus...,An act to add and repeal Section 4788 of the P...


time: 2.35 s (started: 2025-12-12 11:35:37 +00:00)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     1237 non-null   object
 1   summary  1237 non-null   object
 2   title    1237 non-null   object
dtypes: object(3)
memory usage: 29.1+ KB
time: 99.6 ms (started: 2025-12-12 11:35:40 +00:00)


### 🦾 Exercise (optional): Save each row as a .csv file and read them with a wildcard

In [ ]:
# try it!

time: 401 µs (started: 2025-12-12 11:35:40 +00:00)


# 🤗Let's try some model inference from Hugging Face models
I'm obsessed with Hugging Face these days - let's have some fun!
* [tutorial on text pipelines](https://huggingface.co/docs/transformers/pipeline_tutorial#text-pipeline)


# 🔵 Summarization

In [ ]:
from transformers import pipeline
import nltk
nltk.download('punkt')

corpus = df['summary']
#corpus = ' '.join(str(line) for line in df.select('text')) # shorten text
print(corpus)

[nltk_data] Downloading package punkt to /root/nltk_data...


0       Existing property tax law establishes a vetera...
1       Existing law provides that the Board of Parole...
2       The Sales and Use Tax Law imposes a tax on ret...
3       Existing law requires all moneys, except for f...
4       Existing law defines a request regarding resus...
                              ...                        
1232    Existing law, the Carpenter-Presley-Tanner Haz...
1233    (1) The Hazardous Waste Control Law authorizes...
1234    Under existing law, any employer or other pers...
1235    Existing law requires the Director of the Depa...
1236    Existing federal law, the Indian Gaming Regula...
Name: summary, Length: 1237, dtype: object
time: 228 ms (started: 2025-12-12 11:35:40 +00:00)


[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# # let's convert to a list
corpus = corpus.values.tolist()

time: 13.2 ms (started: 2025-12-12 11:35:40 +00:00)


In [ ]:
corpus[0] # actual

'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxation Code, no a

time: 11.4 ms (started: 2025-12-12 11:35:40 +00:00)


In [ ]:
from transformers import pipeline, AutoTokenizer, BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

summarizer = pipeline("summarization",  model = model, tokenizer = tokenizer)
summarizer(corpus[0]) # predicted


Device set to use cpu


[{'summary_text': 'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation. The bill would provide that the exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes. It also would not apply to any portion of a property that consists of a bar where alcohol is served.'}]

time: 25.6 s (started: 2025-12-12 11:46:12 +00:00)


In [ ]:
corpus[0] # actual

'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxation Code, no a

time: 2.48 ms (started: 2025-12-12 11:25:15 +00:00)


^ that was for a single row, we can do the entire dataframe or a few rows (shown below)

### Customizing the call

In [ ]:
inputs = tokenizer(
    corpus[0],
    truncation=True,
    max_length=1000, # play with this!
    return_tensors="pt"
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=250, # play with this!
    min_length=5,
    num_beams=1, # more beams, more time - but more creative responses
    early_stopping=True
)

tokenizer.decode(summary_ids[0], skip_special_tokens=True)

The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Existing property tax law establishes a veterans’ organization exemption. Bill would provide that the exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes.'

time: 14.3 s (started: 2025-12-12 12:03:05 +00:00)


### Is it running locally? Or doing an API call?

The model is downloaded and running locally on your Colab session. The `from_pretrained` method in the transformers library handles downloading the model and its tokenizer to your local runtime if they aren't already cached. The `Device set to use cpu` message also confirms it's running on the Colab instance's CPU, not making API calls to Hugging Face for each inference.

### Tokens, tokens, tokens.
Since the model is running locally, you don't have to worry about API call costs associated with token usage. This means you can process text without incurring external charges per token.

However, it's still important to be mindful of 'tokens' in the context of the model's internal processing. Large language models have a maximum sequence length (often expressed in tokens) that they can handle in a single pass. If your input text exceeds this limit, the model or tokenizer will typically truncate it or raise an error.

So, while cost is no longer a concern, efficiency and the model's architectural limits regarding input size (measured in tokens) still apply.

### Run on first three rows

In [ ]:
# first three rows!
myresults = summarizer(corpus[0:3])

time: 1min 11s (started: 2025-12-12 11:25:15 +00:00)


In [ ]:
myresults # predicted

[{'summary_text': 'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation. The bill would provide that the exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes. It also would not apply to any portion of a property that consists of a bar where alcohol is served.'},
 {'summary_text': 'Existing law provides that the Board of Parole Hearings or its successor in interest shall be the state’s parole authority. Existing law exempts a prisoner sentenced to death or a term of life without the possibility of parole from eligibility for compassionate release. This bill would additionally exempt from medical parole eligibility and compassionate release eligibility a prisoner who was convicted of the first-degree murder of a peace officer.'},
 {'summary_text': 'The bill would expand the application of the Sales and Use Tax law by imposing a tax on specified serv

time: 3.19 ms (started: 2025-12-12 11:26:27 +00:00)


In [ ]:
# actual
corpus[0:3]

['Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxation Code, no 

time: 2.36 ms (started: 2025-12-12 11:26:27 +00:00)


### 🦾 Excercise: Limit the number of tokens
* [limit tokens in pyspark (optional)](https://stackoverflow.com/questions/52975567/get-first-n-elements-from-dataframe-arraytype-column-in-pyspark)

Try to summarize the text by keeping the first N elements (1000?)

# 🔵 Zero Shot Learning
Without knowing anything about the classification problem at hand, can we guess what category this is?

In [ ]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


time: 23.9 s (started: 2025-12-12 11:26:27 +00:00)


In [ ]:
res = classifier(corpus[0],
                 candidate_labels=['good for veterans', 'bad for veterans'])

time: 8.27 s (started: 2025-12-12 11:26:50 +00:00)


In [ ]:
print(res['labels'])
print(res['scores']) # scroll right to see % predicted per class!

['good for veterans', 'bad for veterans']
[0.801446259021759, 0.19855369627475739]
time: 849 µs (started: 2025-12-12 11:26:59 +00:00)


### 🦾 Excercise: Add some different categories and apply to entire dataset
For example, which laws pertain to water, alcohol or neither? Manually check your work! Compare with summarization?

# 🔵 Sentiment Analysis
* More details here: [blog post on sentiment analysis](https://huggingface.co/blog/sentiment-analysis-python)

#### POS vs. NEG




In [ ]:
classifier = pipeline('sentiment-analysis', model = 'distilbert-base-uncased-finetuned-sst-2-english')
res = classifier(corpus[0])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


time: 7.7 s (started: 2025-12-12 12:08:06 +00:00)


In [ ]:
print(res)

[{'label': 'NEGATIVE', 'score': 0.989589273929596}]
time: 2.06 ms (started: 2025-12-12 12:08:26 +00:00)


# 🔵 Emotion Classification with a Multi-label Model

To get a wider range of emotions, we can use a model specifically trained for emotion classification. One popular choice is `distilbert-base-uncased-emotion` or `cardiffnlp/twitter-roberta-base-emotion`. Let's try `cardiffnlp/twitter-roberta-base-emotion`.

In [ ]:
from transformers import pipeline

# Initialize a classifier with a model trained for emotion detection
emotion_classifier = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-emotion', top_k=4)

# Classify the emotion of the first summary in the corpus
# Correctly extract the string from the PySpark DataFrame
res_emotion = emotion_classifier(corpus[0])
print(res_emotion)

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


[[{'label': 'anger', 'score': 0.39260202646255493}, {'label': 'joy', 'score': 0.31325656175613403}, {'label': 'sadness', 'score': 0.18216778337955475}, {'label': 'optimism', 'score': 0.1119735911488533}]]
time: 14.6 s (started: 2025-12-12 11:27:04 +00:00)


You should now see a result with a different set of labels, representing various emotions, along with their scores. This demonstrates how changing the model allows for more granular sentiment or emotion analysis.

In [ ]:
from transformers import pipeline

# Initialize a classifier with a model trained for emotion detection
emotion_classifier = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-emotion', top_k=2)

# Classify the emotion of the first summary in the corpus
# Correctly extract the string from the PySpark DataFrame
res_emotion = emotion_classifier(corpus[0])
print(res_emotion)

Device set to use cpu


[[{'label': 'anger', 'score': 0.39260202646255493}, {'label': 'joy', 'score': 0.31325656175613403}]]
time: 2.88 s (started: 2025-12-12 11:27:36 +00:00)


### 🦾Exercise: See if you can add different models here and get different results (can you get other emotions besides POS and NEG depending on what model you choose?)

time: 3.91 s (started: 2025-12-12 11:27:36 +00:00)


# 🦾 (Optional) Try to read from a Spark dataframe directly

Of course, it's easy to change this to a Spark dataframe.
* [convert to pyspark if you are brave](https://www.geeksforgeeks.org/how-to-convert-pandas-to-pyspark-dataframe/)

In [ ]:
from datasets import load_dataset

billsum = load_dataset("billsum", split="ca_test")

time: 1.67 s (started: 2025-12-12 11:27:40 +00:00)


In [ ]:
# attributes of the dataset
billsum

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 1237
})

time: 2.97 ms (started: 2025-12-12 11:27:42 +00:00)


In [ ]:
# convert to a pandas dataframe for something familiar
# you could always save as a .csv and load dataframe
import pandas as pd
df = pd.DataFrame(billsum)
df.head()

,text,summary,title
0,The people of the State of California do enact...,Existing property tax law establishes a vetera...,An act to amend Section 215.1 of the Revenue a...
1,The people of the State of California do enact...,Existing law provides that the Board of Parole...,"An act to amend Section 3550 of, and to add Se..."
2,The people of the State of California do enact...,The Sales and Use Tax Law imposes a tax on ret...,An act\nto add Chapter 3.8 (commencing with Se...
3,The people of the State of California do enact...,"Existing law requires all moneys, except for f...","An act to amend Sections 75220, 75221, and 752..."
4,The people of the State of California do enact...,Existing law defines a request regarding resus...,An act to add and repeal Section 4788 of the P...


time: 391 ms (started: 2025-12-12 11:27:42 +00:00)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1237 entries, 0 to 1236
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     1237 non-null   object
 1   summary  1237 non-null   object
 2   title    1237 non-null   object
dtypes: object(3)
memory usage: 29.1+ KB
time: 15.2 ms (started: 2025-12-12 11:27:42 +00:00)


## Spark

In [ ]:
# from  pyspark library import
# SparkSession
from pyspark.sql import SparkSession

# Building the SparkSession and name
# it :'pandas to spark'
spark = SparkSession.builder.appName(
  "pandas to spark").getOrCreate()

time: 16.1 s (started: 2025-12-12 11:27:42 +00:00)


In [ ]:
# create DataFrame
df = spark.createDataFrame(df)

df.show()

+--------------------+--------------------+--------------------+
|                text|             summary|               title|
+--------------------+--------------------+--------------------+
|The people of the...|Existing property...|An act to amend S...|
|The people of the...|Existing law prov...|An act to amend S...|
|The people of the...|The Sales and Use...|An act\nto add Ch...|
|The people of the...|Existing law requ...|An act to amend S...|
|The people of the...|Existing law defi...|An act to add and...|
|The people of the...|The Political Ref...|An act to amend S...|
|The people of the...|The Song-Brown He...|An act\nto add Ar...|
|The people of the...|Existing law requ...|An act to add Sec...|
|The people of the...|The California Pu...|An act to amend S...|
|The people of the...|(1) Existing law ...|An act to add Cha...|
|The people of the...|Existing law requ...|An act to add Sec...|
|The people of the...|Existing law prov...|An act to amend S...|
|The people of the...|Exi

In [ ]:
# prompt: keep first three rows of the pyspark dataframe

df = df.limit(3)
df.show()

+--------------------+--------------------+--------------------+
|                text|             summary|               title|
+--------------------+--------------------+--------------------+
|The people of the...|Existing property...|An act to amend S...|
|The people of the...|Existing law prov...|An act to amend S...|
|The people of the...|The Sales and Use...|An act\nto add Ch...|
+--------------------+--------------------+--------------------+

time: 778 ms (started: 2025-12-12 11:28:06 +00:00)


In [ ]:
corpus = df.select('summary')
#corpus = ' '.join(str(line) for line in df.select('text')) # shorten text
print(corpus.show())

+--------------------+
|             summary|
+--------------------+
|Existing property...|
|Existing law prov...|
|The Sales and Use...|
+--------------------+

None
time: 814 ms (started: 2025-12-12 11:28:06 +00:00)


In [ ]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

Device set to use cpu


time: 1.58 s (started: 2025-12-12 11:28:07 +00:00)


In [ ]:
# prompt: i want to use my zero shot classification model on the first row of corpus

res = classifier(corpus.first()[0],
                 candidate_labels=['good for veterans', 'bad for veterans'])
print(res)


{'sequence': 'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxati

In [ ]:
corpus

DataFrame[summary: string]

time: 16.8 ms (started: 2025-12-12 11:28:22 +00:00)


In [ ]:
corpus.show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                             summary|
+----------------------------------------------------------------------------------------------------+
|Existing property tax law establishes a veterans’ organization exemption under which property is ...|
|Existing law provides that the Board of Parole Hearings or its successor in interest shall be the...|
|The Sales and Use Tax Law imposes a tax on retailers measured by the gross receipts from the sale...|
+----------------------------------------------------------------------------------------------------+

time: 577 ms (started: 2025-12-12 11:28:22 +00:00)


## Solution 1 - use `i` in the collect statement!
From my former student Vinni Datla

In [ ]:
myResults = []
for i in range(corpus.count()):

  res = classifier(corpus.select('summary').collect()[i],
                 candidate_labels=['good for veterans', 'bad for veterans'])
  print(res)
  myResults.append(res)


{'sequence': 'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxati

In [ ]:
myResults

[{'sequence': 'Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.\nThis bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.\nSection 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.\nThis bill would provide that, notwithstanding Section 2229 of the Revenue and Taxat

time: 5.11 ms (started: 2025-12-12 11:28:52 +00:00)


In [ ]:
# ready for analysis!
myResults = pd.DataFrame(myResults)
myResults

,sequence,labels,scores
0,Existing property tax law establishes a vetera...,"[good for veterans, bad for veterans]","[0.801446259021759, 0.19855369627475739]"
1,Existing law provides that the Board of Parole...,"[good for veterans, bad for veterans]","[0.5735539197921753, 0.4264460802078247]"
2,The Sales and Use Tax Law imposes a tax on ret...,"[good for veterans, bad for veterans]","[0.6640530824661255, 0.33594685792922974]"


time: 16.9 ms (started: 2025-12-12 11:28:52 +00:00)


## Solution 2- Use a UDF
From my former student, Faeze Safari.

Using a UDF (user defined function!) This is a little advanced but we will cover in future lectures. This example does it for sentiment analysis - but you could also do it for zero shot learning.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

from transformers import pipeline

spark = SparkSession.builder.getOrCreate()

# sentiment-analysis classifier
classifier = pipeline('sentiment-analysis')

# Select a subset of data
def corpus_subset(text):
    return text[:512]

@udf(returnType=StringType()) # Register the sentiment analysis function as a UDF

def classify_sentiment(text):
    corpus_sub = corpus_subset(text)
    result = classifier(corpus_sub)[0]
    # return result['label'] # update me!
    return result

# Apply the UDF to the DataFrame to add a new column with the sentiment analysis results
corpus = corpus.withColumn("sentiment", classify_sentiment(corpus["summary"]))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


time: 3.77 s (started: 2025-12-12 11:28:52 +00:00)


In [ ]:
# this adds a new column to the dataframe!
corpus.printSchema()

root
 |-- summary: string (nullable = true)
 |-- sentiment: string (nullable = true)

time: 4.71 ms (started: 2025-12-12 11:28:56 +00:00)


In [ ]:
# but can you also store the score :)
# you probably need to update the result you are storing!
corpus.show()

+--------------------+--------------------+
|             summary|           sentiment|
+--------------------+--------------------+
|Existing property...|{score=0.55774861...|
|Existing law prov...|{score=0.97598105...|
|The Sales and Use...|{score=0.99873715...|
+--------------------+--------------------+

time: 23.6 s (started: 2025-12-12 11:28:56 +00:00)
